In [1]:
import pandas as pd
import requests
import requests
import re
import json
from bs4 import BeautifulSoup
from concurrent.futures import ThreadPoolExecutor, as_completed

In [ ]:
data_actornames = pd.read_csv("../data/name.basics.tsv", sep="\t")
data_actornames = data_actornames.drop(columns=["deathYear", "birthYear"])
data_actornames = data_actornames[data_actornames["primaryProfession"].str.contains("actor|actress", na=False)]
data_actornames = data_actornames[data_actornames["knownForTitles"].str.contains(",")]
#data_actornames.to_csv("../data/actor_names.csv", index=False)
#data_actornames.to_parquet("../data/name.basics.parquet", engine="pyarrow", index=False)

In [ ]:
def invert_mapping(
    df: pd.DataFrame,
    id_col: str = "nconst",
    titles_col: str = "knownForTitles",
    null_value: str = "\\N",
) -> pd.DataFrame:
    result = df[[id_col, titles_col]].dropna()
    result = result[result[titles_col] != null_value]

    result = result.copy()
    result[titles_col] = result[titles_col].str.split(",")
    result = result.explode(titles_col)
    result[titles_col] = result[titles_col].str.strip()
    result = result[result[titles_col] != ""]

    movie_df = (
        result.groupby(titles_col)[id_col]
        .apply(lambda x: ",".join(sorted(x.unique())))
        .reset_index()
    )
    movie_df.columns = ["tconst", "personIds"]

    return movie_df.sort_values("tconst").reset_index(drop=True)


In [5]:
movie_list = invert_mapping(data_actornames)
movie_list = movie_list[movie_list["personIds"].str.contains(",")]

In [6]:
title_basics = pd.read_csv("../data/title.basics.tsv", sep="\t")
title_basics = title_basics[title_basics["titleType"] == "movie"]
title_basics["startYear"] = pd.to_numeric(title_basics["startYear"], errors="coerce")
title_basics = title_basics[(title_basics["startYear"] > 1957) & (title_basics["startYear"] <= 2026)]
title_basics = title_basics[~title_basics["genres"].str.contains("Documentary", na=False)]
title_basics = title_basics.drop(columns=["titleType", "isAdult", "endYear", "runtimeMinutes", "genres"])

In [21]:
merge_movielist = movie_list.merge(title_basics, on="tconst", how="inner")
#merge_movielist.to_csv("../data/movie_data.csv", index=False)
#merge_movielist.to_parquet("../data/movie_data.parquet", engine="pyarrow", index=False)

In [6]:
movie_list = pd.read_parquet("../data/movie_data.parquet", engine="pyarrow")

In [4]:
TMDB_API_KEY = "cc0093b5ad09a876190e097a1c2c8e65"

def is_english_movie(imdb_id: str) -> dict:
    tmdb_url = f"https://api.themoviedb.org/3/find/{imdb_id}"
    params = {
        "api_key": TMDB_API_KEY,
        "external_source": "imdb_id"
    }

    try:
        response = requests.get(tmdb_url, params=params)
        response.raise_for_status()
        data = response.json()

        movies = data.get("movie_results", [])
        if not movies:
            return {"tconst": imdb_id, "original_language": None}
        
        movie = movies[0]
        original_language = movie.get("original_language")

        return {
            "tconst": imdb_id,
            "original_language": original_language
        }
    
    except requests.exceptions.RequestException as e:
        return {"tconst": imdb_id, "original_language": None}

In [ ]:
def check_movies(df: pd.DataFrame, imdb_id_col: str = "imdb_id", max_workers: int = 20) -> pd.DataFrame:
    imdb_ids = df[imdb_id_col].tolist()
    results = [None] * len(imdb_ids)
    completed = 0
    last_checkpoint = 0

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(is_english_movie, imdb_id): i for i, imdb_id in enumerate(imdb_ids)}

        for future in as_completed(futures):
            idx = futures[future]
            completed += 1
            results[idx] = future.result()

            if completed % 100 == 0:
                print(f"Progress: {completed}/{len(imdb_ids)}")

            if completed - last_checkpoint >= 25000:
                checkpoint_df = pd.DataFrame([r for r in results if r is not None])
                checkpoint_file = f"checkpoint_{completed}.csv"
                checkpoint_df.to_csv(checkpoint_file, index=False)
                print(f"Checkpoint saved at {completed} records -> {checkpoint_file}")
                last_checkpoint = completed

    # Final save for any remaining records
    results_df = pd.DataFrame(results)
    results_df.to_csv("checkpoint_final.csv", index=False)
    print(f"Final checkpoint saved -> checkpoint_final.csv")

    return results_df

#check_movies(movie_list, imdb_id_col="tconst", max_workers=20)

movie_origins = pd.read_csv("../data/checkpoint.csv")

merged_list = movie_list.merge(movie_origins, left_on="tconst", right_on="tconst", how="left")

english_movies = merged_list[merged_list["original_language"] == "en"].drop(columns=["primaryTitle", "original_language"])
#english_movies.to_parquet("../data/english_movies.parquet", index=False)

In [9]:
english_movies = pd.read_parquet("../data/english_movies.parquet", engine="pyarrow")

In [ ]:
def scrape_boxoffice(tconst: str):
    try:
        response = requests.get(f"https://www.boxofficemojo.com/title/{tconst}/")
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")
        performance_summary = soup.find("div", class_="mojo-performance-summary-table")
        boxoffice = performance_summary.find_all("span", string=re.compile(r'.Worldwide*'))
        if boxoffice:
            return boxoffice[0].parent.find_all("span", class_="money")[0].text.strip()
        
    except Exception as e:
        print(f"Error scraping {tconst}: {e}")
        return None

In [11]:
scrape_boxoffice(english_movies["tconst"].iloc[0])

'$76,019,048'

In [12]:
def get_boxoffices(id_list: list) -> pd.DataFrame:
    results = []
    with ThreadPoolExecutor(max_workers=8) as executor:
        futures = [executor.submit(scrape_boxoffice, id) for id in id_list]
        for future in as_completed(futures):
            results.append(future.result())
    return pd.DataFrame({"tconst": id_list, "boxoffice": results})